In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error as mse, r2_score

In [12]:
from sklearn.datasets import fetch_california_housing

# Load dataset
housing = fetch_california_housing()

X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target # Median house value in $100,000s

X.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [3]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MedInc      20640 non-null  float64
 1   HouseAge    20640 non-null  float64
 2   AveRooms    20640 non-null  float64
 3   AveBedrms   20640 non-null  float64
 4   Population  20640 non-null  float64
 5   AveOccup    20640 non-null  float64
 6   Latitude    20640 non-null  float64
 7   Longitude   20640 non-null  float64
dtypes: float64(8)
memory usage: 1.3 MB


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=20)

In [5]:
california = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

california.fit(X_train, y_train)

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None


In [6]:
prediction = california.predict(X_test)

mean_error = mse(
    y_test,
    prediction
)

print(f"Mean Squared Error : {mean_error:.2f}")

Mean Squared Error : 0.54


In [7]:
# The coefficient of determination or R^2

california.score(X_test, y_test)

0.6111963365478914

## Prediction Function

In [37]:
def predict_house_price(my_model, MedInc, HouseAge, AveRooms, AveBedrms,
 Population, AveOccup, Latitude, Longitude):

 # Prepare input as a 2D array (one row, 8 features)
 sample = pd.DataFrame({
 'MedInc': [MedInc],
 'HouseAge': [HouseAge],
 'AveRooms': [AveRooms],
 'AveBedrms': [AveBedrms],
 'Population': [Population],
 'AveOccup': [AveOccup],
 'Latitude': [Latitude],
 'Longitude': [Longitude]
 })

 predicted_price = my_model.predict(sample)[0]
 return f"Estimated House Price: ${predicted_price * 100_000:,.2f}"

In [38]:
predict_house_price( my_model=california,MedInc=8.3, HouseAge=41.0, AveRooms=6.0, AveBedrms=1.0, 
                    Population=980, AveOccup=2.5, Latitude=37.5, Longitude=-122.3)

'Estimated House Price: $438,514.65'

In [39]:
predict_house_price(my_model=california, MedInc=8.3, HouseAge=45.0, AveRooms=6.0, AveBedrms=1,
                     Population=780, AveOccup=2.5, Latitude=37.5, Longitude=-122.3)

'Estimated House Price: $442,428.19'

## RandomForestRegressor

In [29]:
pipeline = Pipeline([
    ("model", RandomForestRegressor(random_state=30))
])

pipeline.fit(X_train, y_train)

,steps,"[('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0


In [30]:
forest_pred = pipeline.predict(X_test)

forest_mean = mse(
    y_test,
    forest_pred
)
print("Forest Mean Squared Error", forest_mean)

Forest Mean Squared Error 0.26571753503089013


In [31]:
pipeline.score(X_test, y_test)

0.8070823785568653

In [32]:
forest_mse = mse(y_test, forest_pred)
forest_r2 = r2_score(y_test, forest_pred)

print("MSE:", forest_mse)
print("R²:", forest_r2)

MSE: 0.26571753503089013
R²: 0.8070823785568653


## Hyperparameter Tuning

In [33]:
forest_param = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}
forest_grid = GridSearchCV(
    estimator=pipeline,
    param_grid=forest_param,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=1
)
forest_grid.fit(X_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


,estimator,Pipeline(step...m_state=30))])
,param_grid,"{'model__max_depth': [None, 20], 'model__min_samples_leaf': [1, 2], 'model__min_samples_split': [2, 5], 'model__n_estimators': [100, 200]}"
,scoring,'neg_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,200


In [34]:
print("Best Parameters:", forest_grid.best_params_)
print("Best CV MSE:", -forest_grid.best_score_)

Best Parameters: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 200}
Best CV MSE: 0.26346367962499306


In [35]:
best_model = forest_grid.best_estimator_

tuned_pred = best_model.predict(X_test)

tuned_mean = mse(
    y_test,
    tuned_pred
)
print("Tuned MSE", tuned_mean)

Tuned MSE 0.262793815818434


In [36]:
tuned_r2 = r2_score(y_test, tuned_pred)

print("Tuned R²:", r2_score(y_test, tuned_pred))

Tuned R²: 0.809205072327034


## Prediction Function for ForestRegressor

In [40]:
def predict_house_price(my_model, MedInc, HouseAge, AveRooms, AveBedrms,
 Population, AveOccup, Latitude, Longitude):

 # Prepare input as a 2D array (one row, 8 features)
 sample = pd.DataFrame({
 'MedInc': [MedInc],
 'HouseAge': [HouseAge],
 'AveRooms': [AveRooms],
 'AveBedrms': [AveBedrms],
 'Population': [Population],
 'AveOccup': [AveOccup],
 'Latitude': [Latitude],
 'Longitude': [Longitude]
 })

 predicted_price = my_model.predict(sample)[0]
 return f"Estimated House Price: ${predicted_price * 100_000:,.2f}"

In [42]:
predict_house_price( my_model=best_model,MedInc=8.3, HouseAge=41.0, AveRooms=6.0, AveBedrms=1.0, 
                    Population=980, AveOccup=2.5, Latitude=37.5, Longitude=-122.3)

'Estimated House Price: $466,423.60'